# Discrete Shell Demo
Demonstrate the discrete shell bending energy available in from the `mesh_energy` module.

In [ ]:
import MeshFEM
import mesh, mesh_energy
import viewer
import numpy as np

In [ ]:
import igl
m = mesh.Mesh('../3rdparty/MeshFEM/misc/examples/meshes/square.off')
m = mesh.Mesh(*igl.upsample(m.vertices(), m.elements(), 5), embeddingDimension=3)

In [ ]:
optVars = mesh_energy.NodalVars(m, 3) # Create per-node position variables (the variable dimension 3 here can also be inferred).
em = MeshFEM.EmbeddedMesh(m, optVars) # A wrapper object used to visualize the deformation described by `optVars`.

In [ ]:
v = viewer.Viewer(em, wireframe=True)
v.show()

In [ ]:
neo = mesh_energy.NeoHookeanMembrane(m, optVars)
dsb = mesh_energy.DiscreteShellBending(m, optVars)

In [ ]:
# Set up Dirichlet boundary conditions.
clampVtx = m.vertices()[:, 0] < -0.9
pullDownVtx = m.vertices()[:, 0] > 0.95

clampedVerts = np.where(clampVtx)[0]
pullDownVerts = np.where(pullDownVtx)[0]
fixedVars = list(3 * clampedVerts) + list(3 * clampedVerts + 1) + list(3 * clampedVerts + 2) + list(3 * pullDownVerts + 2)

In [ ]:
# Set initial z coordinate linearly interpolated from 0 at the left to -0.5 at the right.
a = -0.9
b = 1.0
z = -0.5 * np.clip((m.vertices()[:, 0] - a) / (b - a), 0, 1)

In [ ]:
# This line is needed to assign a per-element material property field.
# (By default, all elements share the same material object.)
dsb.allocatePerElementMaterials()

In [ ]:
# Set spatially varying bending hinge stiffness based on midpoint x coordinate (interpolating from stiff at the left to flexible at the right).
edges = []
m.visitEdges(lambda e, ei: edges.append(sorted(e)))
boundaryEdges = np.sort(m.boundaryElements(), axis=1)
innerEdges = np.array([e for e in {tuple(e) for e in edges} - {tuple(e) for e in boundaryEdges}])
edge_midpt_x = m.vertices()[innerEdges].mean(axis=1)[:, 0]
alpha = np.clip((edge_midpt_x - a) / (b - a), 0, 1)
stiffness = 1 * alpha + 2 * (1 - alpha)

for i, k in enumerate(stiffness):
    dsb.materialForElement(i).stiffness = k

In [ ]:
V = m.vertices()
V[:, 2] = z
optVars.setVars(V.ravel())

In [ ]:
import py_newton_optimizer

p = py_newton_optimizer.NewtonMultiobjectiveProblem(optVars, [neo, dsb])
p.setFixedVars(fixedVars)
opt = p.optimizer()

opt.optimize()
v.update()